# resist. — YOLOv8 Training (Single Model)

**Architecture note:**  
The jbhepner dataset labels each band bounding box with the actual color as its class name.  
So ONE YOLO model handles both detection AND color identification. No CNN needed.

**What this notebook does:**
1. Downloads jbhepner/resistor-and-band-detection from Roboflow (1,285 real photos)
2. Normalises class names (bronze→brown, grey→gray, purple→violet)
3. Trains YOLOv8n to output color-labelled bounding boxes
4. Validates and visualises predictions
5. Exports to ONNX and saves 2 files to Google Drive

**Before you start:** Runtime → Change runtime type → T4 GPU  
**Estimated time:** ~20 minutes

---

## Step 0 — Install and setup

In [ ]:
!pip install -q ultralytics roboflow onnx onnxruntime
print('Packages installed')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/resist_models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Drive mounted. Saving to: {SAVE_DIR}')

In [ ]:
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('No GPU found. Go to Runtime > Change runtime type > T4 GPU')

---
## Step 1 — Download the dataset

Get your free Roboflow API key:
1. Sign up at roboflow.com
2. Profile icon > Settings > Roboflow API > copy Private API Key

Dataset: jbhepner/resistor-and-band-detection  
1,285 real resistor photos. Bands labelled by color. CC BY 4.0 license.

In [ ]:
# PASTE YOUR ROBOFLOW API KEY HERE
ROBOFLOW_API_KEY = 'YOUR_KEY_HERE'

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('jbhepner').project('resistor-and-band-detection')
dataset = project.version(6).download('yolov8')

DATASET_PATH = dataset.location
print(f'Dataset downloaded to: {DATASET_PATH}')

In [ ]:
import os, yaml
data_yaml = os.path.join(DATASET_PATH, 'data.yaml')
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)

print('Classes:', cfg.get('names'))
print('Count:  ', cfg.get('nc'))
train_dir = os.path.join(DATASET_PATH, 'train', 'images')
valid_dir = os.path.join(DATASET_PATH, 'valid', 'images')
print('Train images:', len(os.listdir(train_dir)))
print('Val images:  ', len(os.listdir(valid_dir)))

---
## Step 2 — Normalise class names

The dataset uses bronze/grey/purple. We remap them to the standard resistor  
color names (brown/gray/violet) and drop the non-color classes (resistor, breadboard).

In [ ]:
import glob

with open(data_yaml) as f:
    cfg = yaml.safe_load(f)
original_classes = cfg['names']
print('Original:', original_classes)

# Remap dataset class names to standard resistor color names
COLOR_MAP = {
    'black': 'black', 'brown': 'brown', 'bronze': 'brown',
    'red': 'red', 'orange': 'orange', 'yellow': 'yellow',
    'green': 'green', 'blue': 'blue', 'purple': 'violet',
    'grey': 'gray', 'white': 'white', 'gold': 'gold', 'silver': 'silver',
}
SKIP = {'resistor', 'breadboard'}

seen = []
old_to_new = {}
for idx, name in enumerate(original_classes):
    if name in SKIP:
        old_to_new[idx] = None
        continue
    new_name = COLOR_MAP.get(name, name)
    if new_name not in seen:
        seen.append(new_name)
    old_to_new[idx] = seen.index(new_name)

cfg['names'] = seen
cfg['nc'] = len(seen)
with open(data_yaml, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Updated:', seen)
print('Index map:', old_to_new)

In [ ]:
# Patch all label .txt files with new class indices
label_dirs = [os.path.join(DATASET_PATH, s, 'labels')
              for s in ['train', 'valid', 'test']
              if os.path.exists(os.path.join(DATASET_PATH, s, 'labels'))]

dropped = relabelled = 0
for label_dir in label_dirs:
    for txt in glob.glob(os.path.join(label_dir, '*.txt')):
        lines = open(txt).readlines()
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts: continue
            new_cls = old_to_new.get(int(parts[0]))
            if new_cls is None: dropped += 1; continue
            if new_cls != int(parts[0]): relabelled += 1
            new_lines.append(f'{new_cls} {" ".join(parts[1:])}\n')
        open(txt, 'w').writelines(new_lines)

print(f'Labels patched. Dropped: {dropped} | Re-indexed: {relabelled}')
print(f'Final classes ({len(seen)}):', seen)

---
## Step 3 — Train YOLOv8n

Using 100 epochs with stronger augmentation to compensate for the smaller dataset size.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=25,
    device=0,
    project='/content/runs',
    name='resist_detector',
    exist_ok=True,
    # Augmentation — important with only 1.2k images
    hsv_h=0.015,   # slight hue shift
    hsv_s=0.5,     # saturation variation
    hsv_v=0.4,     # brightness variation
    fliplr=0.5,    # horizontal flip
    mosaic=1.0,    # combine 4 images (great for small datasets)
    mixup=0.1,
    verbose=True,
)

BEST_PT = '/content/runs/resist_detector/weights/best.pt'
print(f'Training complete. Best weights: {BEST_PT}')

In [ ]:
# Validate
val_model = YOLO(BEST_PT)
metrics = val_model.val(data=data_yaml, device=0)
print(f'mAP50:    {metrics.box.map50:.3f}  (good = >0.65)')
print(f'mAP50-95: {metrics.box.map:.3f}')

if metrics.box.map50 >= 0.65:
    print('Accuracy looks good - ready to deploy')
elif metrics.box.map50 >= 0.50:
    print('Decent - will work in good lighting')
else:
    print('Below target - consider adding more training images')

In [ ]:
# Visual check
import glob
from IPython.display import Image, display

val_imgs = (glob.glob(os.path.join(DATASET_PATH, 'valid', 'images', '*.jpg')) +
            glob.glob(os.path.join(DATASET_PATH, 'valid', 'images', '*.png')))

pred_model = YOLO(BEST_PT)
for img in val_imgs[:5]:
    pred_model(img, save=True, project='/content/preview', name='val', exist_ok=True)
for img in glob.glob('/content/preview/val/*.jpg')[:5]:
    display(Image(filename=img, width=500))

---
## Step 4 — Export to ONNX and save to Drive

In [ ]:
import json, shutil
import numpy as np
import onnx, onnxruntime as ort

# Export YOLO to ONNX
export_model = YOLO(BEST_PT)
export_model.export(format='onnx', dynamic=True, simplify=True)

ONNX_PATH = '/content/band_detector.onnx'
shutil.copy(BEST_PT.replace('.pt', '.onnx'), ONNX_PATH)

# Verify
sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
dummy = np.zeros((1,3,640,640), dtype=np.float32)
out = sess.run(None, {sess.get_inputs()[0].name: dummy})
print(f'ONNX verified. Output shape: {out[0].shape}')
print(f'File size: {os.path.getsize(ONNX_PATH)/1024/1024:.1f} MB')

In [ ]:
# Save class list
with open(data_yaml) as f:
    final_cfg = yaml.safe_load(f)

CLASSES_JSON = '/content/band_classes.json'
with open(CLASSES_JSON, 'w') as f:
    json.dump(final_cfg['names'], f)
print('Classes saved:', final_cfg['names'])

In [ ]:
# Copy to Google Drive
for src, fname in [(ONNX_PATH, 'band_detector.onnx'), (CLASSES_JSON, 'band_classes.json')]:
    dst = os.path.join(SAVE_DIR, fname)
    shutil.copy(src, dst)
    print(f'Saved {fname} to Drive ({os.path.getsize(dst)/1024:.0f} KB)')

print('\nDone! Download these 2 files from Google Drive > resist_models/')
print('Place them in: inference/models/')

---
## Step 5 — End-to-end smoke test

In [ ]:
import cv2, numpy as np, json
import onnxruntime as ort
from IPython.display import Image, display

DIGIT = dict(black=0,brown=1,red=2,orange=3,yellow=4,green=5,blue=6,violet=7,gray=8,white=9)
MULT  = dict(black=1,brown=10,red=100,orange=1000,yellow=10000,green=100000,
             blue=1000000,violet=10000000,gray=100000000,white=1000000000,gold=0.1,silver=0.01)
TOL   = dict(brown='+-1%',red='+-2%',green='+-0.5%',blue='+-0.25%',
             violet='+-0.1%',gray='+-0.05%',gold='+-5%',silver='+-10%')

with open(CLASSES_JSON) as f:
    CLASS_NAMES = json.load(f)

sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
inp  = sess.get_inputs()[0].name

def detect(img_path, conf=0.35):
    img  = cv2.imread(img_path)
    h, w = img.shape[:2]
    scale = 640 / max(h, w)
    nh, nw = int(h*scale), int(w*scale)
    pad = np.full((640,640,3), 114, np.uint8)
    pad[:nh,:nw] = cv2.resize(img,(nw,nh))
    x = np.transpose(pad[:,:,::-1].astype(np.float32)/255,(2,0,1))[None]

    raw = sess.run(None,{inp:x})[0]
    p = raw[0].T
    nc = len(CLASS_NAMES)
    scores = p[:,4:4+nc].max(1)
    clsids = p[:,4:4+nc].argmax(1)
    mask = scores > conf
    if not mask.any(): print('No bands detected'); return

    boxes = p[mask,:4]
    xyxy  = np.clip(np.stack([
        (boxes[:,0]-boxes[:,2]/2)/scale, (boxes[:,1]-boxes[:,3]/2)/scale,
        (boxes[:,0]+boxes[:,2]/2)/scale, (boxes[:,1]+boxes[:,3]/2)/scale
    ],1), 0, [w,h,w,h])

    # Simple NMS
    sc = scores[mask]; cl = clsids[mask]
    order = sc.argsort()[::-1]; keep=[]
    while order.size:
        i=order[0]; keep.append(i)
        if order.size==1: break
        ix1=np.maximum(xyxy[i,0],xyxy[order[1:],0])
        iy1=np.maximum(xyxy[i,1],xyxy[order[1:],1])
        ix2=np.minimum(xyxy[i,2],xyxy[order[1:],2])
        iy2=np.minimum(xyxy[i,3],xyxy[order[1:],3])
        inter=np.maximum(0,ix2-ix1)*np.maximum(0,iy2-iy1)
        a0=(xyxy[i,2]-xyxy[i,0])*(xyxy[i,3]-xyxy[i,1])
        ao=(xyxy[order[1:],2]-xyxy[order[1:],0])*(xyxy[order[1:],3]-xyxy[order[1:],1])
        order=order[1:][inter/(a0+ao-inter+1e-9)<0.45]

    kb=xyxy[keep]; kc=cl[keep]
    lr=np.argsort((kb[:,0]+kb[:,2])/2)
    bands=[CLASS_NAMES[kc[i]] for i in lr]
    print(f'Bands L->R: {bands}')

    vis=img.copy()
    for i in lr:
        x1,y1,x2,y2=kb[i].astype(int)
        cv2.rectangle(vis,(x1,y1),(x2,y2),(0,200,255),2)
        cv2.putText(vis,CLASS_NAMES[kc[i]],(x1,y1-6),cv2.FONT_HERSHEY_SIMPLEX,0.45,(0,200,255),1)
    out='/content/smoke.jpg'; cv2.imwrite(out,vis); display(Image(out,width=600))

    try:
        n=len(bands)
        if n==4:
            print(f'  = {(DIGIT[bands[0]]*10+DIGIT[bands[1]])*MULT[bands[2]]} ohms  {TOL.get(bands[3],"")}')
        elif n==5:
            print(f'  = {(DIGIT[bands[0]]*100+DIGIT[bands[1]]*10+DIGIT[bands[2]])*MULT[bands[3]]} ohms  {TOL.get(bands[4],"")}')
    except: pass

for p in (glob.glob(os.path.join(DATASET_PATH,'valid','images','*.jpg'))+
          glob.glob(os.path.join(DATASET_PATH,'valid','images','*.png')))[:3]:
    print(f'\n{os.path.basename(p)}')
    detect(p)

---
## What to download from Google Drive

| File | Purpose |
|---|---|
| `band_detector.onnx` | The trained model (~6 MB) |
| `band_classes.json`  | Class name list |

Copy to your project at `inference/models/`.  
Flask auto-detects them on startup and switches to offline mode.